# 03 - Build Silver/Gold curated tables

Promotes `bronze.*` to curated tables with clean business names in the **`gold`**
schema (`gold.dim_*`, `gold.fact_*`) and casts date/timestamp columns for the SQL
endpoint. The **`silver`** schema is created but left empty for now (the synthetic
source is already clean, so no separate silver transformations are needed yet).
ML tables and `customer_360` are built in notebook 04.

In [ ]:
from pyspark.sql import functions as F

spark.sql('CREATE SCHEMA IF NOT EXISTS silver')  # intentionally empty for now
spark.sql('CREATE SCHEMA IF NOT EXISTS gold')

curated = ['dim_geography', 'dim_product', 'dim_plan', 'dim_device', 'dim_promotion', 'dim_customer', 'dim_account', 'fact_subscription', 'fact_invoice', 'fact_invoice_line', 'fact_usage_data', 'fact_usage_voice', 'fact_coverage', 'fact_outage', 'fact_service_metric', 'fact_work_order', 'fact_appointment', 'fact_contact', 'fact_offer', 'fact_feedback']
date_cols = {'dim_account': ['open_date', 'close_date'], 'dim_customer': ['date_of_birth'], 'fact_subscription': ['start_date', 'end_date'], 'fact_invoice': ['period_start', 'period_end', 'due_date', 'paid_date'], 'fact_offer': ['presented_date'], 'fact_feedback': ['feedback_date'], 'fact_work_order': ['opened_date', 'closed_date'], 'fact_usage_data': ['usage_date'], 'fact_usage_voice': ['usage_date'], 'fact_service_metric': ['metric_date']}
ts_cols = {'fact_outage': ['start_time', 'end_time'], 'fact_contact': ['contact_ts']}

for t in curated:
    df = spark.table(f'bronze.{t}')
    for c in date_cols.get(t, []):
        if c in df.columns:
            df = df.withColumn(c, F.to_date(F.col(c)))
    for c in ts_cols.get(t, []):
        if c in df.columns:
            df = df.withColumn(c, F.to_timestamp(F.col(c)))
    (df.write.format('delta').mode('overwrite')
        .option('overwriteSchema', 'true').saveAsTable(f'gold.{t}'))
    print(f'gold.{t:24s} {df.count():>8,} rows')

In [ ]:
print('Curated gold dim/fact tables built. silver schema is empty by design.')